In [1]:
import ReportTimerange as rtr
import ReportMailer as rm
import GuestbookDbConnect as gdb

## Change History ##

- __09/19/24__  
    - 1) Complete accounting of check-ins, showers, laundry, and clothing closet services used, and credits earned by day for the past7days.
    - 2) Dates of every check-in for the past 30days.
    - 3) Total number of check-ins, showers, laundry, and clothing closet visits, credits earned, and each  housing category over the past 30 days .
    - 4) Total number of check-ins for the past 365days.
    - 5) Date of first check-in
    
    
- __10/25/24__ 
- 1) added check for SERVICE_COMPLETED in listing of servicerequests

# Install a conda package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install xlsxwriter

## Reporting Periods ##
* PD - yesterday
* PW - previous week
* PM - previous month
* PQ - previous quarter
* PY - previous calendar year
* YTD year to date
* WTD current week to date
* TODAY today
* P90 - previous 90days

In [2]:
import os
import pandas as pd

path = "/home/gbadmin/jupy-notebooks/Reports/inqueue"
dir_list = os.listdir(path)
print(dir_list)

if len(dir_list)>0:
  filename = "/home/gbadmin/jupy-notebooks/Reports/inqueue/{}".format(dir_list[0])
  print(filename)

  rep = pd.read_json(filename, lines=True)

  for _ , row in rep.iterrows():    
    _INTERVAL = row['interval']
    _TYPE = row['type']
    if (_TYPE=='IND'):
       _CLIENTID = row['client']
       _APPUSER  = row['appuser']
       print(_INTERVAL)
       print(_TYPE)
       print(_CLIENTID)
       print("AppUser {}".format(_APPUSER))
       idperson = _CLIENTID
       # delete the file
       os.remove(filename)
    else:
      print("Non-ID Type")
      exit()
       
else:
  print("NOTHING TO DO")
  exit()

['client-report-20261010_095923-charlie.json']
/home/gbadmin/jupy-notebooks/Reports/inqueue/client-report-20261010_095923-charlie.json
YTD
IND
3539
AppUser 4


idperson = 814
_INTERVAL = "YTD"
_APPUSER = 1

In [3]:
from pandas import ExcelWriter
from pandas import ExcelFile
import xlsxwriter
#import datetime
from datetime import datetime

def createSpreadsheetAndMailIt(collections, reportname, recipients, subject, body):
  now = datetime.now().strftime('%Y-%m-%d_%H%M%S')
  filename = reportname + "-" + now + ".xlsx"
  #writer = ExcelWriter(filename)
  writer = ExcelWriter(filename, engine='xlsxwriter')
  workbook = writer.book  
  print("Writing dataframe to Excel file {0}".format(filename))
  for collection in collections:
    print("Writing {} to spreadsheet".format(collection["name"]))
    collection["dataframe"].to_excel(writer, sheet_name=collection["name"], index=False)
    worksheet = writer.sheets[collection["name"]]
    if 'colwidths' in collection:
      colwidths = collection['colwidths']
      #print("colwidths={}".format(colwidths))
      for n in range(0, len(colwidths)):
        col = str(chr(65 + n))
        worksheet.set_column('{}:{}'.format(col,col), colwidths[n])
    else:
      worksheet.set_column('A:A', 30)
  writer.close()
  cc_list = []
  for recipient in recipients: 
    rm.mailer(recipient, subject, body, filename, cc_list)
  print("Excel/Email Done!")

In [4]:
import psycopg2 as pg
import pandas.io.sql as psql
import pandas as pd
import GuestbookDbConnect as gdb

conn = gdb.guestbookDbConnect()

collections = []
summaries = []

Welcome to Jupyter Notebook.  You are connected to the Opportunity House guestbook database!


## Get Client Name for Report ##

In [5]:
query = "SELECT firstname, lastname, idperson FROM guestbook_person WHERE idperson= {}".format(idperson)

data = pd.read_sql(query, conn)

firstname = data.iloc[0,0]
lastname  = data.iloc[0,1]
fullname = "{} {}".format(firstname,lastname)

print(fullname)


CHARLES WILLIS


/tmp/ipykernel_696172/3626315863.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


## Table of Contents ##

In [6]:
import ReportTimerange as rtr

trange = rtr.timerange(_INTERVAL)
print(trange)

collection = {}
collection["name"] = "Contents"

df = pd.DataFrame({_INTERVAL + " Report Tab for " + fullname :[ \
              "Client Summary", \
              "Personal Info", \
              "7 Day Activity", \
              "30 Day Check-Ins", \
              "30 Day Activity", \
              _INTERVAL + " Visits by Sub-Int", \
              _INTERVAL + " Services by Sub-Int", \
              _INTERVAL + " Suspensions"
              ], \
                   "Description":[ \
              "line item rollups", \
              "client personal information", \
              "visits, services by date prior 7days", \
              "visits by date prior 30days", \
              "services provided prior 30days", \
              "visits by sub-interval of reporting period", \
              "services by sub-interval of reporting period", \
              "suspension history in reporting period"
                                 ]})

collection["dataframe"] = df 
collection["colwidths"] = [30, 100]
collections.append(collection)

df.head(1000)

[datetime.datetime(2026, 1, 1, 0, 0), datetime.datetime(2026, 10, 10, 9, 59, 54, 862570), 'YYYY-MM']


,YTD Report Tab for CHARLES WILLIS,Description
0,Client Summary,line item rollups
1,Personal Info,client personal information
2,7 Day Activity,"visits, services by date prior 7days"
3,30 Day Check-Ins,visits by date prior 30days
4,30 Day Activity,services provided prior 30days
5,YTD Visits by Sub-Int,visits by sub-interval of reporting period
6,YTD Services by Sub-Int,services by sub-interval of reporting period
7,YTD Suspensions,suspension history in reporting period


## Personal Info##

In [7]:
from datetime import datetime
now = datetime.now(tz=None)

collection = {}
collection["name"] = "Personal Info"

query = "SELECT firstname, lastname, ethnicity_id, gender_id, \
         language_id, veteran_id, credits \
         FROM guestbook_person WHERE idperson={}".format(idperson)

data = pd.read_sql(query, conn)

#print(data)

firstname = data.iloc[0,0]
lastname  = data.iloc[0,1]
fullname = "{} {}".format(firstname,lastname)
ethnicity_id  = data.iloc[0,2]
gender_id    = data.iloc[0,3]
language_id  = data.iloc[0,4]
veteran_id   = data.iloc[0,5]
credits   = data.iloc[0,6]

print("Report for {} {}".format(firstname, lastname))
print("client ID is {}\n".format(idperson))

if (ethnicity_id==1):
    ethnicity  = "Unknown"
elif (ethnicity_id==2):
    ethnicity  = "Asian"
elif (ethnicity_id==3):
     ethnicity  = "Black"
elif (ethnicity_id==4):
      ethnicity  = "Hispanic"
elif (ethnicity_id==5):
     ethnicity  = "Native American"
elif (ethnicity_id==6):
      ethnicity  = "White"
        
if (gender_id==1):
    gender = "Unknown"
elif (gender_id==2):
    gender = "Male"
elif (gender_id==3):
    gender = "Female"
        
if (language_id==1):
   language = "Preferred language unknown"
elif (language_id==2):
   language = "Preferred language is English"
elif (language_id==3):
   language = "Preferred language is Spanish"
        
if (veteran_id==1):
    veteran = "Veteran status is unknown"
elif (veteran_id==2):
    veteran = "Is a veteran of the US Armed Forces"
elif (veteran_id==3):
    veteran = "Is not a veteran of the US Armed Forces"
        
print("Client has {} credits in the OpportunityHouse Bank\n".format(credits))

query = "SELECT min(timestamp)AS firstdate, max(timestamp) AS lastdate \
         FROM guestbook_personsnapshot \
         WHERE person_id={}".format(idperson)

data = pd.read_sql(query, conn)

firstvisit = "{} ({} days ago)".format(data.iloc[0,0].strftime('%m/%d/%Y'), (now- data.iloc[0,0].replace(tzinfo=None)).days)
lastvisit  = "{} ({} days ago)".format(data.iloc[0,1].strftime('%m/%d/%Y'), (now- data.iloc[0,1].replace(tzinfo=None)).days)

df = pd.DataFrame({"Attribute":[ \
              "Name", \
              "Client ID", \
              "Ethnicity", \
              "Gender", \
              "Language", \
              "Veteran", \
              "First Check-In", \
              "Last Check-In", \
              "OppHouse credits"
              ], \
                   "Value":[ \
              fullname, \
              idperson, \
              ethnicity, \
              gender, \
              language, \
              veteran, \
              firstvisit, \
              lastvisit, \
              credits
              ]})

collection["dataframe"] = df 
collection["colwidths"] = [30, 100]
collections.append(collection)

print(df)

df.head(1000)

Report for CHARLES WILLIS
client ID is 3539

Client has 7 credits in the OpportunityHouse Bank

          Attribute                                    Value
0              Name                           CHARLES WILLIS
1         Client ID                                     3539
2         Ethnicity                                    White
3            Gender                                   Female
4          Language            Preferred language is English
5           Veteran  Is not a veteran of the US Armed Forces
6    First Check-In                06/15/2026 (116 days ago)
7     Last Check-In                 09/10/2026 (29 days ago)
8  OppHouse credits                                        7


/tmp/ipykernel_696172/2218889559.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)
/tmp/ipykernel_696172/2218889559.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,Attribute,Value
0,Name,CHARLES WILLIS
1,Client ID,3539
2,Ethnicity,White
3,Gender,Female
4,Language,Preferred language is English
5,Veteran,Is not a veteran of the US Armed Forces
6,First Check-In,06/15/2026 (116 days ago)
7,Last Check-In,09/10/2026 (29 days ago)
8,OppHouse credits,7


# Trailing 7day Detail ##
added 09/2024

1) Complete accounting of check-ins, showers, laundry, and clothing closet services used, and credits earned by day for the past week. Organize this data in the most convenient way for you.

added 10/2024

2) PersonServiceRequest has field 'status' to record delivery of the service.
* SERVICE_STATUS_QUEUED          = 500
* SERVICE_STATUS_COMPLETED       = 501
* SERVICE_STATUS_NOTCOMPLETED    = 502
* SERVICE_STATUS_REMOVED         = 503

3) We may need to treat MEALS separately as they are not usually confirmed by staff. Are there other services we don't confirm?

4) Services rendered by the client have service.points > 0

In [8]:
collection = {}
collection["name"] = "7 Day Activity"

query = "SELECT \
         TO_CHAR(timestamp, 'MM/DD/YYYY') AS checkin_date, left(service.name, strpos(service.name, '/') - 1) AS services FROM guestbook_personservicerequest servicerequest\
         JOIN guestbook_personsnapshot snapshot ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE (( service.idservice!=1 AND servicerequest.status=501) OR (service.idservice=1) ) AND snapshot.person_id = {}  AND timestamp>NOW() - INTERVAL '7 DAYS' \
         ORDER BY TIMESTAMP desc".format(idperson)

#print(query)
data = pd.read_sql(query, conn)
#print(data)

data2 = []
dt = None

lasttimestamp = 0
if (data.shape[0]>0):
  for n in range(0, data.shape[0]):  
    timestamp = data.iloc[n,0]
    servicename = data.iloc[n,1]
    if (timestamp == lasttimestamp):
      servicenames = servicenames + " " + servicename        
    else:
      if (dt!=None):
        dt["services"] = servicenames
        data2.append(dt)
      dt = {'date': timestamp, 'services': []}
      servicenames = ""
      servicenames = servicenames + " " + servicename
      lasttimestamp = timestamp
      pass

  dt["services"] = servicenames
  data2.append(dt)
else:
    print("Nothing to report")
    
df = pd.DataFrame(data2)
collection["dataframe"] = df 
collection["colwidths"] = [30,300]
collections.append(collection)

# Print the data
print(df)

Nothing to report
Empty DataFrame
Columns: []
Index: []


/tmp/ipykernel_696172/2909712676.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


## 30day Check-Ins by Date ##
added  09/2024

In [9]:
# 2) Dates of every check-in for the last 30 days.

collection = {}
collection["name"] = "30 Day Check-Ins"

query = "SELECT TO_CHAR(timestamp, 'MM/DD/YYYY') AS checkin_date FROM guestbook_personsnapshot snapshot \
        WHERE snapshot.person_id={} AND timestamp> NOW() - INTERVAL '30 DAYS' \
        ORDER BY timestamp desc".format(idperson)         

#print(query)
data = pd.read_sql(query, conn)

collection["dataframe"] = data 
collection["colwidths"] = [30]
collections.append(collection)
data.head(1000)
#print(collection)

/tmp/ipykernel_696172/415913108.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,checkin_date
0,09/10/2026


## 30day Activity ##

added  09/2024

* Services delivered to client
* Housing

In [10]:
# 3) Total number of check-ins, showers, laundry, and clothing closet visits, 
#credits earned, and each housing category over the past 30 days 
#(just total for each like the current summary page).

lst = []

q0 = "SELECT min(timestamp)AS first_checkin, max(timestamp) AS last_checkin, \
         COUNT(DISTINCT idsnapshot) AS total_checkins FROM guestbook_personsnapshot \
         WHERE person_id={} AND timestamp>NOW() - INTERVAL '30 DAYS'".format(idperson)

#print(query)
d0 = pd.read_sql(q0, conn)

summary = {}
summary["name"]  = "Check-Ins"
summary["count"] = d0.iloc[0,2]
print("{} {}".format(summary["name"],summary["count"]))
summaries.append(summary)

lst.append(["Check-Ins", d0.iloc[0,2]])

query = "SELECT \
         left(service.name, strpos(service.name, '/') - 1) AS service_name, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE ((service.idservice!=1 AND servicerequest.status=501) OR (service.idservice=1)) AND snapshot.person_id = {} AND timestamp>NOW() - INTERVAL '30 DAYS' AND service.points<=0 \
         GROUP BY service_name \
         ORDER BY total desc".format(idperson)


#print(query)
data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = data.iloc[n,0]
  summary["count"] = data.iloc[n,1]
  print("{} {}".format(summary["name"],summary["count"]))
  summaries.append(summary)

  lst.append([str(data.iloc[n,0]), data.iloc[n,1]])    

q2 = "SELECT  \
         left(housing.name, strpos(housing.name, '(') - 1)  AS housing_type, COUNT(*) AS nights_spent FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
         WHERE snapshot.person_id={} AND timestamp>NOW() - INTERVAL '30 DAYS' AND prompt_id=8 \
         GROUP BY housing_type".format(idperson)

d2 = pd.read_sql(q2, conn)

for n in range(0, d2.shape[0]):  
  summary = {}
  summary["name"]  = "Housing " + str(d2.iloc[n,0])
  summary["count"] = d2.iloc[n,1]
  print("{} {}".format(summary["name"],summary["count"]))
  summaries.append(summary)

  lst.append(["Housing " + str(d2.iloc[n,0]), d2.iloc[n,1]])

df = pd.DataFrame(lst, columns = ["30 Day Activity", "Value"])

print(df)

collection = {}
collection["name"] = "30 Day Activity"
collection["dataframe"] = df 
collection["colwidths"] = [50, 10]
collections.append(collection)

Check-Ins 1
Meal 1
Housing Sheltered  1
      30 Day Activity  Value
0           Check-Ins      1
1                Meal      1
2  Housing Sheltered       1


/tmp/ipykernel_696172/533867217.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  d0 = pd.read_sql(q0, conn)
/tmp/ipykernel_696172/533867217.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)
/tmp/ipykernel_696172/533867217.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  d2 = pd.read_sql(q2, conn)


## Trailing Year Check-In Count ##
added 09/2024

In [11]:
#4) Total number of check-ins for the past one year.
#5) Date of first check-in
now = datetime.now(tz=None)

query = "SELECT COUNT(DISTINCT timestamp) FROM guestbook_personsnapshot snapshot \
        WHERE snapshot.person_id={} AND timestamp>NOW() - INTERVAL '1 YEAR'  \
        ".format(idperson)         

#print(query)
data = pd.read_sql(query, conn)

trailingYearCheckins = data.iloc[0,0]

query = "SELECT min(timestamp)AS firstdate, max(timestamp) AS lastdate \
         FROM guestbook_personsnapshot \
         WHERE person_id={}".format(idperson)

data = pd.read_sql(query, conn)

firstvisit = "{} ({} days ago)".format(data.iloc[0,0].strftime('%m/%d/%Y'), (now- data.iloc[0,0].replace(tzinfo=None)).days)

#df = pd.DataFrame({"Attribute":["Trailing Year Check-Ins", "First Check-In"], "Value":[trailingYearCheckins,firstvisit]})

summary = {}
summary["name"]  = "First Check-In"
summary["count"] = firstvisit
summaries.append(summary)

summary = {}
summary["name"]  = "365 Day Check-Ins"
summary["count"] = trailingYearCheckins
summaries.append(summary)

df.head(1000)

/tmp/ipykernel_696172/3981473511.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)
/tmp/ipykernel_696172/3981473511.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,30 Day Activity,Value
0,Check-Ins,1
1,Meal,1
2,Housing Sheltered,1


## Total visits in report period ##

In [12]:
query = "SELECT min(timestamp)AS first_checkin, max(timestamp) AS last_checkin, \
         COUNT(DISTINCT idsnapshot) AS total_checkins FROM guestbook_personsnapshot \
         WHERE person_id={} AND timestamp BETWEEN '{}' AND '{}'".format(idperson, trange[0], trange[1])

#print(query)
data = pd.read_sql(query, conn)

summary = {}
summary["name"] = _INTERVAL + " Check-Ins"
summary["count"] = data.iloc[0,2]
summaries.append(summary)
#print(summaries)

data.head(100)

/tmp/ipykernel_696172/1850432739.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,first_checkin,last_checkin,total_checkins
0,2026-06-15 11:36:20.579585+00:00,2026-09-10 18:46:24.869659+00:00,2


## Visits by Sub-Interval ##
* Statistics focused on the regularity of client visits over the reporting period.

In [13]:
collection = {}
collection["name"] = "Checkins by Sub-Int"

query = "SELECT to_char(timestamp,'{}') AS period,  \
         COUNT(DISTINCT snapshot.idsnapshot) AS checkins \
         FROM guestbook_personsnapshot snapshot \
         JOIN guestbook_person person ON person.idperson={} \
         WHERE snapshot.person_id = {} AND timestamp BETWEEN '{}' AND '{}' \
         GROUP BY period".format(trange[2], idperson, idperson, trange[0], trange[1])

#print(query)
data = pd.read_sql(query, conn)

persons = data.pivot_table('checkins', index=['period']).fillna(0).astype(int).reset_index('period')

collection["dataframe"] = persons 
collection["colwidths"] = [30, 20]
collections.append(collection)

persons.head(1000)

/tmp/ipykernel_696172/4187790385.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,period,checkins
0,2026-06,1
1,2026-09,1


## Services Delivered to Client ##

In [14]:
query = "SELECT \
         left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE (( service.idservice!=1 AND servicerequest.status=501) OR (service.idservice=1) ) AND snapshot.person_id = {} AND timestamp BETWEEN '{}' AND '{}'  AND service.points<=0 \
         GROUP BY servicename \
         ORDER BY total desc".format(idperson, trange[0], trange[1])

#print(query)
data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = _INTERVAL + " " + data.iloc[n,0] + " services delivered."
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
    

data.head(1000)

/tmp/ipykernel_696172/2340278600.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,servicename,total
0,Meal,2
1,Clothin,1
2,Showe,1


## Services by Sub-Interval ##

In [15]:
collection = {}
collection["name"] = "Services by Sub-Int"

query = "SELECT to_char(timestamp,'{}') AS period, \
         left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE snapshot.person_id={} AND servicerequest.status=501 AND timestamp BETWEEN '{}' AND '{}' AND service.points<=0 \
         GROUP BY period, servicename".format(trange[2], idperson, trange[0], trange[1])

#print(query)
data = pd.read_sql(query, conn)

if (not data.empty):
  services = data.pivot_table('total', index=['period'], columns='servicename').fillna(0).astype(int).reset_index('period')

  collection["dataframe"] = services 
  collection["colwidths"] = [30, 20, 20, 20, 20, 20, 20, 20]
  collections.append(collection)

  services.head(1000)

/tmp/ipykernel_696172/893761200.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


## Housing ##
*summary line entry*

In [16]:
query = "SELECT  \
         left(housing.name, strpos(housing.name, '(') - 1)  AS housing_type, COUNT(*) AS nights_spent FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
         WHERE snapshot.person_id={} AND timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
         GROUP BY housing_type".format(idperson, trange[0], trange[1])

#print(query)

data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = _INTERVAL + " Nights in " + data.iloc[n,0] 
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)

data.head(1000)

/tmp/ipykernel_696172/3473730295.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,housing_type,nights_spent
0,Sheltered,1
1,Unsheltered,1


# Suspensions #

In [17]:
collection = {}
collection["name"] = "Suspension History"

query = "SELECT starttime AS suspension_date, (endtime-starttime) AS days, suspensionreason.name AS reason FROM guestbook_suspension \
         JOIN guestbook_suspensionreason AS suspensionreason ON idsuspensionreason = reason_id \
         WHERE person_id={} AND starttime BETWEEN '{}' AND '{}'".format(idperson, trange[0], trange[1])

data = pd.read_sql(query, conn)

collection["dataframe"] = data
collections.append(collection)

data.head(1000)

/tmp/ipykernel_696172/3069261076.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,suspension_date,days,reason


## Incarcerations ##
*summary line entry*

In [18]:

query = "SELECT COUNT(DISTINCT(idsnapshot)) as incarcerations \
         FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         WHERE snapshot.person_id={} AND survey.object_id=2 AND timestamp BETWEEN '{}' AND '{}' \
         AND prompt_id=4".format(idperson, trange[0], trange[1])

#print(query)

data = pd.read_sql(query, conn)

summary = {}
summary["name"]  = _INTERVAL + " Incarcerations" 
summary["count"] = data.iloc[0,0]
summaries.append(summary)

data.head(1000)

/tmp/ipykernel_696172/1181301553.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,incarcerations
0,0


## Client Summary ##

In [19]:
collection = {}
summ = pd.DataFrame.from_dict(summaries)
collection["name"] = "Client Summary"
collection["dataframe"] = summ
collection["colwidths"] = [30, 60]
collections.insert(1, collection)
#collections.append(collection)
summ.head(1000)

,name,count
0,Check-Ins,1
1,Meal,1
2,Housing Sheltered,1
3,First Check-In,06/15/2026 (116 days ago)
4,365 Day Check-Ins,2
5,YTD Check-Ins,2
6,YTD Meal services delivered.,2
7,YTD Clothin services delivered.,1
8,YTD Showe services delivered.,1
9,YTD Nights in Sheltered,1


In [20]:
acronyms = [ "PD", "PW", "PM", "PQ", "PY", "YTD", "WTD", "TODAY", "P90"]
friendly = [ "Prior Day", "Prior Week", "Prior Month", "Prior Quarter", "Prior Year", "Year to Date", "Week to Date", "Today", "Previous 90 days"]

def toPeriodFriendly(INTERVAL):
    for n in range(0, len(acronyms)):
      if INTERVAL==acronyms[n]:
         return friendly[n]
    return "Unknown Period"

## Spreadsheet Generator ##
*and emailer*

In [21]:
import ReportTimerange as rtr
import ReportMailer as rm

#import os
#import pandas as pd
#import GuestbookDbConnect as gdb
#conn = gdb.guestbookDbConnect()

_REPORTNAME = _INTERVAL
_SUBJECT = "Opportunity House - Client Report  for {} {}, {}".format(firstname, lastname, toPeriodFriendly(_INTERVAL))
_BODY    = "Spreadsheet (attached) with reports for period {} through {}.".format(trange[0], trange[1])

#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com', 'jahood1@yahoo.com']
#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com', 'pastor@opphouse.net']
#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com']

query = "SELECT idappuser, email FROM guestbook_appuser WHERE idappuser={}".format(_APPUSER)

#print(query)
data = pd.read_sql(query, conn)
_EMAIL_RECIPIENT = {data.iloc[0,1]}
#_EMAIL_RECIPIENT = ["holobox@gmail.com"]
print(">>{}<<".format(_EMAIL_RECIPIENT))

createSpreadsheetAndMailIt(collections, _REPORTNAME, _EMAIL_RECIPIENT, _SUBJECT, _BODY)

/tmp/ipykernel_695998/2504162638.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


>>{'holobox@gmail.com'}<<
Writing dataframe to Excel file YTD-2026-10-10_095636.xlsx
Writing Contents to spreadsheet
Writing Client Summary to spreadsheet
Writing Personal Info to spreadsheet
Writing 7 Day Activity to spreadsheet
Writing 30 Day Check-Ins to spreadsheet
Writing 30 Day Activity to spreadsheet
Writing Checkins by Sub-Int to spreadsheet
Writing Services by Sub-Int to spreadsheet
Writing Suspension History to spreadsheet


TypeError: mailer() missing 1 required positional argument: 'cc_list'